In [12]:
import os
import time
import json
import random
import warnings
from itertools import combinations
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold,LeaveOneOut
from sklearn.neighbors import KNeighborsClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

os.chdir("") #set to your working directory

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# =========================
# Configuration
# =========================
INPUT_CSV = "./ICGCfilterREDO.csv"
OUTPUT_ROOT = "./output" #change to your desired output name
SEED = 42


pd.set_option("display.max_rows", 10)
pd.set_option("display.max_columns", 10)

random.seed(SEED)
np.random.seed(SEED)
loo = LeaveOneOut()


def safe_name(text: str) -> str:
    """Create a filesystem-friendly folder/file component."""
    return (
        text.replace(" ", "_")
        .replace("/", "-")
        .replace("\\", "-")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "")
    )


def make_predictions_df(
    true_labels: pd.Series,
    proba_predictions: np.ndarray,
    samples: pd.DataFrame,
    reverse_mapping: Dict[int, str],
) -> pd.DataFrame:
    true_named = true_labels.map(reverse_mapping)
    true_named.name = "subtype"

    proba_df = pd.DataFrame(proba_predictions)
    if proba_df.shape[1] == len(reverse_mapping):
        proba_df = proba_df.rename(columns=reverse_mapping)
    else:
        proba_df.columns = [f"class_{i}" for i in range(proba_df.shape[1])]

    proba_df.index = samples["id"].values
    true_named.index = samples["id"].values
    predictions = pd.concat([true_named, proba_df], axis=1)
    predictions.index.name = "id"
    return predictions


def multiclass_sensitivity_specificity_jaccard_gmeasure(cm: np.ndarray) -> Tuple[float, float, float, float]:
    """Return sensitivity, specificity, jaccard, g_measure as macro averages."""
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        g_measure = np.sqrt(sensitivity * specificity) if sensitivity >= 0 and specificity >= 0 else 0.0
        jaccard = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        return sensitivity, specificity, jaccard, g_measure

    sensitivities = []
    specificities = []
    jaccards = []
    g_measures = []

    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - (fp + fn + tp)

        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        jaccard = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        g_measure = np.sqrt(sensitivity * specificity) if sensitivity >= 0 and specificity >= 0 else 0.0

        sensitivities.append(sensitivity)
        specificities.append(specificity)
        jaccards.append(jaccard)
        g_measures.append(g_measure)

    return (
        float(np.mean(sensitivities)),
        float(np.mean(specificities)),
        float(np.mean(jaccards)),
        float(np.mean(g_measures)),
    )


def evaluate_classifier(
    true_labels: pd.Series,
    predicted_labels: pd.Series,
    proba_predictions: np.ndarray,
    stacked: str,
    model_name: str,
    #randomstate: SEED,
    samples: pd.DataFrame,
    reverse_mapping: Dict[int, str],
    combo_id: int,
    combo_name: str,
    data_name: str,
    output_root: str,
) -> Dict[str, object]:
    #print(proba_predictions)
    true_labels = pd.Series(true_labels).reset_index(drop=True)
    predicted_labels = pd.Series(predicted_labels).reset_index(drop=True)

    predictions_df = make_predictions_df(true_labels, proba_predictions, samples, reverse_mapping)

    accuracy = accuracy_score(true_labels, predicted_labels)
    precision = precision_score(true_labels, predicted_labels, average="weighted", zero_division=0)
    weighted_recall = recall_score(true_labels, predicted_labels, average="weighted", zero_division=0)
    f1 = f1_score(true_labels, predicted_labels, average="weighted", zero_division=0)
    mcc = matthews_corrcoef(true_labels, predicted_labels)

    auc = np.nan
    try:
        if proba_predictions.shape[1] == 2:
            auc = roc_auc_score(true_labels, proba_predictions[:, 1])
        else:
            auc = roc_auc_score(true_labels, proba_predictions, multi_class="ovr", average="weighted")
    except Exception:
        auc = np.nan

    cm = confusion_matrix(true_labels, predicted_labels)
    sensitivity, specificity, jaccard, g_measure = multiclass_sensitivity_specificity_jaccard_gmeasure(cm)

    metrics_row = {
        "combo_id": combo_id,
        "base_combo": combo_name,
        "n_base_models": len(combo_name.split("__")) if combo_name else 0,
        "stage": stacked,
        "model_name": model_name,
        #"randomstate": randomstate,
        "Accuracy": accuracy,
        "Precision": precision,
        "Weighted Recall": weighted_recall,
        "Recall (Sensitivity)": sensitivity,
        "F1 Score": f1,
        "MCC": mcc,
        "G-Measure": g_measure,
        "AUC": auc,
        "Specificity": specificity,
        "Jaccard Index": jaccard,
    }

    metrics_df = pd.DataFrame([metrics_row])

    combo_folder = f"combo_{combo_id:03d}__{combo_name}"
    stage_folder = safe_name(stacked)
    model_folder = safe_name(model_name)
    run_dir = os.path.join(output_root, combo_folder, "val_data", stage_folder, model_folder)
    os.makedirs(run_dir, exist_ok=True)

    metrics_path = os.path.join(run_dir, f"{safe_name(data_name)}({SEED})_metrics.csv")
    predictions_path = os.path.join(run_dir, f"{safe_name(data_name)}({SEED})_predictions.csv")

    metrics_df.to_csv(metrics_path, index=False)
    predictions_df.to_csv(predictions_path)

    return metrics_row

def get_oof(
    xtrain: pd.DataFrame,
    ytrain: pd.Series,
    stacked: str,
    models: List[object],
    model_names: List[str],
    samples: pd.DataFrame,
    reverse_mapping: Dict[int, str],
    combo_id: int,
    combo_name: str,
    data_name: str,
    output_root: str,
    collect_metrics: bool = True,
) -> Tuple[Dict[str, np.ndarray], pd.Series, Dict[str, pd.Series], str, np.ndarray, List[Dict[str, object]]]:
    oof_true_labels = pd.Series(dtype=int)
    oof_original_data = pd.DataFrame()
    oof_pred: Dict[str, pd.Series] = {}
    oof_prob_pred: Dict[str, np.ndarray] = {}

    for fold_idx, (train_index, val_index) in enumerate(loo.split(xtrain, ytrain), start=1):
        kf_x_train = xtrain.iloc[train_index]
        kf_y_train = ytrain.iloc[train_index]
        kf_x_val = xtrain.iloc[val_index]
        kf_y_val = ytrain.iloc[val_index]

        if stacked == "individual_models":
            oof_original_data = pd.concat([oof_original_data, kf_x_val], axis=0, ignore_index=True)

        for idx, model_proto in enumerate(models):
            model_name = model_names[idx]
            model = clone(model_proto)
            model.fit(kf_x_train, kf_y_train)
            proba_val = model.predict_proba(kf_x_val)
            pred_val = np.argmax(proba_val, axis=1)

            if model_name in oof_pred:
                oof_pred[model_name] = pd.concat(
                    [oof_pred[model_name], pd.Series(pred_val)], ignore_index=True
                )
            else:
                oof_pred[model_name] = pd.Series(pred_val)

            if model_name in oof_prob_pred:
                oof_prob_pred[model_name] = np.concatenate([oof_prob_pred[model_name], proba_val], axis=0)
            else:
                oof_prob_pred[model_name] = proba_val

        oof_true_labels = pd.concat([oof_true_labels, kf_y_val.reset_index(drop=True)], ignore_index=True)
        #print(f"{stacked} | fold {fold_idx}/{N_SPLITS} done")
        print(f"{stacked} | loo fold {fold_idx}/{len(xtrain)} done")

    oof_original_data_numpy = oof_original_data.to_numpy()
    metrics_records = []

    if collect_metrics:
        for idx, _ in enumerate(models):
            model_name = model_names[idx]
            metrics_row = evaluate_classifier(
                true_labels=oof_true_labels,
                predicted_labels=oof_pred[model_name],
                proba_predictions=oof_prob_pred[model_name],
                stacked=stacked,
                model_name=model_name,
                samples=samples,
                reverse_mapping=reverse_mapping,
                combo_id=combo_id,
                combo_name=combo_name,
                data_name=data_name,
                output_root=output_root,
            )
            metrics_records.append(metrics_row)

    return oof_prob_pred, oof_true_labels, oof_pred, stacked, oof_original_data_numpy, metrics_records


def workflow(
    x: pd.DataFrame,
    samples: pd.DataFrame,
    base_models: List[object],
    base_model_names: List[str],
    stack_models: List[object],
    stack_model_names: List[str],
    combo_id: int,
    combo_name: str,
    data_name: str,
    output_root: str,
) -> List[Dict[str, object]]:
    start_time = time.time()

    x = x.reset_index(drop=True)
    samples = samples.reset_index(drop=True)
    
    y = x["subtype"].copy()
    features = x.drop(columns=["subtype"]).copy()
    
    features.reset_index(drop=True, inplace=True)
    y.reset_index(drop=True, inplace=True)

    label_values = sorted(y.unique())
    mapping = {label: idx for idx, label in enumerate(label_values)}
    reverse_mapping = {v: k for k, v in mapping.items()}
    y_encoded = y.map(mapping).astype(int)

    print(f"\n=== combo {combo_id:03d}: {combo_name} ===")
    #print(f"Random state: {randomstate}")
    print(f"Base models: {base_model_names}")
    print(f"Stack models: {stack_model_names}")

    # Block A: base models
    
    stacked = "individual_models"
    oof_prob_pred, oof_true_labels, _, _, oof_original_data, _ = get_oof(
        xtrain=features,
        ytrain=y_encoded,
        stacked=stacked,
        models=base_models,
        model_names=base_model_names,
        samples=samples,
        reverse_mapping=reverse_mapping,
        combo_id=combo_id,
        combo_name=combo_name,
        data_name=data_name,
        output_root=output_root,
        collect_metrics=True,)

    # Keep base-model columns in the exact specified order
    base_proba_blocks = [oof_prob_pred[name] for name in base_model_names]
    meta_features = np.concatenate(base_proba_blocks + [oof_original_data], axis=1)
    meta_features = pd.DataFrame(meta_features)

    # Block B: stacked models
    stacked = "stacked"
    _, _, _, _, _, stack_records = get_oof(
        xtrain=meta_features,
        ytrain=oof_true_labels,
        stacked=stacked,
        models=stack_models,
        model_names=stack_model_names,
        #randomstate=randomstate,
        samples=samples,
        reverse_mapping=reverse_mapping,
        combo_id=combo_id,
        combo_name=combo_name,
        data_name=data_name,
        output_root=output_root,
    )

    elapsed = time.time() - start_time
    print(f"combo {combo_id:03d} finished in {elapsed:.2f} seconds")

    return stack_records



def build_model_catalog(seed: int) -> Tuple[List[Tuple[str, object]], List[Tuple[str, object]]]:
    base_catalog = [
        ("svm_rbf", SVC(kernel="rbf", probability=True, break_ties=True, random_state=1, C=1, gamma="scale", class_weight=None)),
        ("svm_linear", SVC(kernel="linear", probability=True, break_ties=True, random_state=1, C=0.00075, class_weight="balanced")),
        ("lr", LogisticRegression(max_iter=700, solver="lbfgs")),
        ("rf", RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=1)),
        ("xgb", XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=seed, subsample=1.0, colsample_bytree=1.0, n_jobs=1)),
        ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance")),
        ("qda", QuadraticDiscriminantAnalysis(reg_param=1.0, store_covariance=True, tol=0.0)),
        ("dt", DecisionTreeClassifier(random_state=SEED)),
        ("mlp", MLPClassifier(activation="relu", alpha=0.0001, learning_rate="constant", max_iter=210, batch_size=8, solver="adam", hidden_layer_sizes=(100, 100), random_state=SEED, shuffle=True)),
        ("nb", GaussianNB())
    ]

    stack_catalog = [
        ("stack_xgb", XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=seed, subsample=1.0, colsample_bytree=1.0, n_jobs=1)),
        ("stack_qda", QuadraticDiscriminantAnalysis(reg_param=1.0, store_covariance=True, tol=0.0)),
        ("stack_knn", KNeighborsClassifier(n_neighbors=5, weights="distance")),
        ("stack_rf", RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=1)),
        ("stack_lr", LogisticRegression(max_iter=700, solver="lbfgs")),
        ("stack_svm_rbf", SVC(kernel="rbf", probability=True, break_ties=True, random_state=1, C=1.5, gamma="scale", class_weight="balanced")),
        ("stack_svm_linear", SVC(kernel="linear", probability=True, break_ties=True, random_state=1)),
        ("stack_dt", DecisionTreeClassifier(random_state=SEED)),
        ("stack_mlp", MLPClassifier(activation="relu", alpha=0.0001, learning_rate="constant", max_iter=210, batch_size=8, solver="adam", hidden_layer_sizes=(100, 100), random_state=SEED, shuffle=True)),
        ("stack_nb", GaussianNB())
    ]
    return base_catalog, stack_catalog



    
    
def generate_base_combinations(base_catalog: List[Tuple[str, object]]) -> List[Tuple[List[str], List[object]]]:
    all_combos: List[Tuple[List[str], List[object]]] = []
    indices = list(range(len(base_catalog)))

    for r in range(10, 11): 
        for idx_tuple in combinations(indices, r):
            combo_names = [base_catalog[i][0] for i in idx_tuple]
            combo_models = [base_catalog[i][1] for i in idx_tuple]
            all_combos.append((combo_names, combo_models))
    return all_combos





    

def main() -> None:
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    x = pd.read_csv(INPUT_CSV)
    data_name = os.path.splitext(os.path.basename(INPUT_CSV))[0]

    if "subtype" not in x.columns:
        raise ValueError("Input CSV must contain a 'subtype' column.")

    if "id" in x.columns:
        sample_ids = x["id"].astype(str).copy()
        feature_table = x.drop(columns=["id"]).copy()
    else:
        sample_ids = pd.Series([f"sample_{i}" for i in range(len(x))], name="id")
        feature_table = x.copy()

    samples = pd.DataFrame({"id": sample_ids})
    samples.index = feature_table.index

    base_catalog, stack_catalog = build_model_catalog(SEED)
    stack_model_names = [name for name, _ in stack_catalog]
    stack_models = [model for _, model in stack_catalog]

    all_base_combos = generate_base_combinations(base_catalog)
    print(f"Total first-layer combinations: {len(all_base_combos)}")

    summary_records: List[Dict[str, object]] = []

    for combo_id, (base_model_names, base_models) in enumerate(all_base_combos, start=1):
        combo_name = "__".join(base_model_names)
        combo_records = workflow(
            x=feature_table,
            samples=samples.copy(),
            base_models=base_models,
            base_model_names=base_model_names,
            stack_models=stack_models,
            stack_model_names=stack_model_names,
            combo_id=combo_id,
            combo_name=combo_name,
            data_name=data_name,
            output_root=OUTPUT_ROOT,
        )
        summary_records.extend(combo_records)

    summary_df = pd.DataFrame(summary_records)
    summary_path = os.path.join(OUTPUT_ROOT, "summary.csv")
    summary_df.to_csv(summary_path, index=False)

    if not summary_df.empty:
        best_idx = summary_df.groupby(["combo_id", "stage"])["MCC"].idxmax()
        best_df = summary_df.loc[best_idx].sort_values(["combo_id", "stage"])
        best_df.to_csv(os.path.join(OUTPUT_ROOT, "best_per_combo_and_stage.csv"), index=False)

        overall_best = summary_df.sort_values("MCC", ascending=False).head(50)
        overall_best.to_csv(os.path.join(OUTPUT_ROOT, "top50_by_mcc.csv"), index=False)

    config = {
        "input_csv": INPUT_CSV,
        "output_root": OUTPUT_ROOT,
        "seed": SEED,
        "cv_type": "LeaveOneOut",
        "n_first_layer_candidates": len(base_catalog),
        "n_first_layer_combinations": len(all_base_combos),
        "stack_models": stack_model_names,
    }
    with open(os.path.join(OUTPUT_ROOT, "run_config.json"), "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print(f"\nDone. Summary saved to: {summary_path}")


if __name__ == "__main__":
    main()


Total first-layer combinations: 1

=== combo 001: svm_rbf__svm_linear__lr__rf__xgb__knn__qda__dt__mlp__nb ===
Base models: ['svm_rbf', 'svm_linear', 'lr', 'rf', 'xgb', 'knn', 'qda', 'dt', 'mlp', 'nb']
Stack models: ['stack_xgb', 'stack_qda', 'stack_knn', 'stack_rf', 'stack_lr', 'stack_svm_rbf', 'stack_svm_linear', 'stack_dt', 'stack_mlp', 'stack_nb']
individual_models | loo fold 1/96 done
individual_models | loo fold 2/96 done
individual_models | loo fold 3/96 done
individual_models | loo fold 4/96 done
individual_models | loo fold 5/96 done
individual_models | loo fold 6/96 done
individual_models | loo fold 7/96 done
individual_models | loo fold 8/96 done
individual_models | loo fold 9/96 done
individual_models | loo fold 10/96 done
individual_models | loo fold 11/96 done
individual_models | loo fold 12/96 done
individual_models | loo fold 13/96 done
individual_models | loo fold 14/96 done
individual_models | loo fold 15/96 done
individual_models | loo fold 16/96 done
individual_model